# Model Implementation and Fine Tuning - AG News Text Classification

The AG News Corpus is a popular dataset for text classification, containing news articles categorized into four classes:

    - World,
    - Sports,
    - Business, and
    - Sci/Tech.
    
### Acceptance Criterion

The final output should be a report that compares the

    - performance,
    - efficiency, and
    - resource usage of selected transformer models.
    
Key performance metrics will be accuracy, F1-score, training/inference time, and model size.


## Imports

In [ ]:
!pip install transformers datasets torch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Import necessary libraries

In [ ]:
# Import necessary libraries
import pandas as pd  # For handling CSV files and data manipulation
import torch  # PyTorch for model training and tensor operations
from torch.utils.data import DataLoader  # For batching and loading data
from torch.optim import AdamW  # Optimizer for fine-tuning
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,  # RoBERTa model and tokenizer
    DistilBertTokenizer, DistilBertForSequenceClassification,  # DistilBERT model and tokenizer
    GPT2Tokenizer, GPT2ForSequenceClassification  # GPT-2 model and tokenizer
)
from datasets import Dataset, DatasetDict  # For handling datasets in Hugging Face format
from tqdm import tqdm  # For progress bars during training and evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  # For performance metrics
import time  # For measuring inference time
import numpy as np  # For numerical operations

import os  # For file and directory operations
import zipfile  # For creating zip archives
#from google.colab import files  # For downloading files from Colab

## Device Configuration

In [ ]:
# Set the device for computation (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load Dataset

In [ ]:
# Load the training and test CSV files into pandas DataFrames
train_df = pd.read_csv("AG_News_Classification/train.csv")
test_df = pd.read_csv("AG_News_Classification/test.csv")

In [ ]:
train_df.head()

,class_index,title,description
0,1,Explosion Rocks Baghdad Neighborhood,"BAGHDAD, Iraq, August 24 -- A car bomb explode..."
1,1,BBC reporters' log,BBC correspondents record events in the Middle...
2,1,Israel welcomes Rice nomination; Palestinians ...,Israel on Tuesday warmly welcomed the naming o...
3,1,Medical Journal Calls for a New Drug Watchdog,Medical researchers said the U.S. needs a syst...
4,1,Militants Kidnap Relatives of Iraqi Minister-TV,Militants have kidnapped two relatives of Iraq...


In [ ]:
train_df.size

60000

In [ ]:
test_df.size

3900

## Process data

#### Combine 'title' and 'description' columns into single column 'text'

In [ ]:
# Combine Title and Description columns into a single text column for each sample
# Assumes CSV has 'Title' and 'Description' columns; modify if different
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

#### Adjust labels to be 0-based

In [ ]:
# Adjust labels to be 0-based (AG News labels are 1–4; subtract 1 to make them 0–3)
# PyTorch expects 0-based class indices
train_df["label"] = train_df["class_index"] - 1
test_df["label"] = test_df["class_index"] - 1

#### Convert pandas DataFrames to Hugging Face Dataset objects

In [ ]:
# Convert pandas DataFrames to Hugging Face Dataset objects
# Keep only 'text' and 'label' columns for training
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

#### Why Convert Pandas DataFrames to Hugging Face Dataset Objects?

- Compatibility with Hugging Face Transformers
    - The Hugging Face transformers library (used for training models like BERT, RoBERTa, etc.) is designed to work seamlessly with Hugging Face's Dataset objects.
- Efficient Data Handling
    - provides memory-efficient storage and fast access to large datasets compared to pandas DataFrames.
- Built-in Features for Machine Learning
    - Shuffling, splitting, and batching: Easily shuffle or split datasets (e.g., train_test_split) or create batches for training.
    - Mapping functions: Apply preprocessing (e.g., tokenization) efficiently across the dataset using dataset.map().
    - Filtering: Filter rows based on conditions (e.g., remove invalid labels).
- Standardized Format for Pipelines
    - Converting to Dataset ensures a standardized format that integrates with Hugging Face's ecosystem, including tokenizers, data loaders, and evaluation metrics.
    - This reduces the need for custom data handling code when working with pandas DataFrames.
- Parallel Processing
    - Dataset objects support parallelized preprocessing (e.g., tokenizing text across multiple CPU cores) via dataset.map(), which is faster than pandas' row-wise operations for large datasets.
- Simplified Workflow for Training
    - When using the Trainer API, Dataset objects can be directly passed to the train_dataset and eval_dataset arguments, streamlining the training process.

#### Create a DatasetDict

In [ ]:
# Create a DatasetDict to hold train and test datasets
datasets = DatasetDict({"train": train_dataset, "test": test_dataset})

#### function to tokenize data for a given tokenizer

In [ ]:
# Define a function to tokenize data for a given tokenizer
def tokenize_function(examples, tokenizer, max_length=512):
    # Tokenize the 'text' column, padding to max_length and truncating
    # Use tokenizer-specific max_length (e.g., 512 for RoBERTa, 256 for DistilBERT, 512 for GPT-2)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)

#### Define a function to evaluate a model on the test set

In [ ]:
# Define a function to evaluate a model on the test set
def evaluate_model(model, dataloader, device, model_name):
    # Set model to evaluation mode (disables dropout, etc.)
    model.eval()
    predictions = []
    true_labels = []
    inference_times = []
    memory_usages = []

    # Disable gradient computation for evaluation to save memory
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f" purest Evaluating {model_name}"):
            # Move input tensors to the device
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            labels = batch["label"].to(device)

            # Measure inference time
            start_time = time.time()
            outputs = model(**inputs)
            inference_times.append(time.time() - start_time)

            # Get predicted class by taking the argmax of logits
            batch_predictions = torch.argmax(outputs.logits, dim=-1)

            # Collect predictions and true labels
            predictions.extend(batch_predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

            # Measure GPU memory usage (if on GPU)
            if device.type == "cuda":
                memory_usages.append(torch.cuda.memory_allocated(device) / 1024**2)  # Convert to MB

    # Compute performance metrics using weighted averages for multi-class classification
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average="weighted")
    recall = recall_score(true_labels, predictions, average="weighted")
    f1 = f1_score(true_labels, predictions, average="weighted")

    # Compute average inference time per batch and total inference time
    avg_inference_time = np.mean(inference_times)
    total_inference_time = np.sum(inference_times)

    # Compute average memory usage (if on GPU)
    avg_memory_usage = np.mean(memory_usages) if memory_usages else None

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_inference_time": avg_inference_time,
        "total_inference_time": total_inference_time,
        "avg_memory_usage": avg_memory_usage
    }

In [ ]:
# Dictionary to store results for comparison
results = {}

### Processing Models

#### RoBERTa Model

In [ ]:
# --- RoBERTa Model ---
# Print a status message to indicate that the RoBERTa model processing is starting.
print("Processing RoBERTa model...")

# - `RobertaTokenizer`: Tokenizer for RoBERTa model from Hugging Face's transformers library.
# - `RobertaForSequenceClassification`: Pre-trained RoBERTa model for sequence classification tasks.
# - `DataLoader`: PyTorch utility to create batches of data for training/testing.
# - `AdamW`: Optimizer for fine-tuning transformer models.
# - `tqdm`: Progress bar for tracking training loops.
# - `device`: Assumed to be a PyTorch device (e.g., 'cuda' or 'cpu') defined elsewhere.

# Initialize RoBERTa tokenizer and model
# Load the pre-trained RoBERTa tokenizer from the 'roberta-base' model.
# The tokenizer converts input text into token IDs compatible with RoBERTa.
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Load the pre-trained RoBERTa model for sequence classification.
# - 'roberta-base': Specifies the base version of RoBERTa (12 layers, 768 hidden size).
# - `num_labels=4`: Configures the model for a classification task with 4 output classes.
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=4)

# Move the model to the specified device (e.g., GPU or CPU).
# This ensures all computations (training/inference) happen on the chosen hardware.
roberta_model.to(device)

# Tokenize datasets for RoBERTa
# Define a tokenize function (assumed to be defined elsewhere) that processes input text.
# - `x`: A batch of dataset examples containing 'text' and 'label' fields.
# - `roberta_tokenizer`: The RoBERTa tokenizer to convert text to token IDs.
# - `max_length=512`: Truncates/pads input sequences to a maximum length of 512 tokens (RoBERTa's limit).
# The `datasets` variable is a Hugging Face DatasetDict with 'train' and 'test' splits.
roberta_tokenized_datasets = datasets.map(lambda x: tokenize_function(x, roberta_tokenizer, max_length=512), batched=True)

# Set the dataset format to PyTorch tensors for compatibility with DataLoader.
# - `format="torch"`: Converts dataset columns to PyTorch tensors.
# - `columns=["input_ids", "attention_mask", "label"]`: Specifies which columns to include as tensors.
#   - `input_ids`: Token IDs for the input text.
#   - `attention_mask`: Binary mask indicating which tokens are real (1) vs. padding (0).
#   - `label`: The target class labels for classification.
roberta_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Create data loaders for RoBERTa
# Create a DataLoader for the training dataset.
# - `roberta_tokenized_datasets["train"]`: The tokenized training split.
# - `batch_size=16`: Processes 16 examples per batch to balance memory usage and training speed.
# - `shuffle=True`: Randomly shuffles the training data at the start of each epoch to improve generalization.
roberta_train_dataloader = DataLoader(roberta_tokenized_datasets["train"], batch_size=16, shuffle=True)

# Create a DataLoader for the test dataset.
# - `roberta_tokenized_datasets["test"]`: The tokenized test split.
# - `batch_size=16`: Uses the same batch size as training for consistency.
# - `shuffle=False`: No shuffling for test data, as order doesn't affect evaluation.
roberta_test_dataloader = DataLoader(roberta_tokenized_datasets["test"], batch_size=16)

# Set up optimizer for RoBERTa
# Initialize the AdamW optimizer, commonly used for fine-tuning transformer models.
# - `roberta_model.parameters()`: Specifies the model's trainable parameters.
# - `lr=2e-5`: Sets a learning rate of 2e-5, a typical value for fine-tuning transformers.
roberta_optimizer = AdamW(roberta_model.parameters(), lr=2e-5)

# Training loop for RoBERTa
# Define the number of training epochs (iterations over the entire training dataset).
num_epochs = 3

# Iterate over the specified number of epochs.
for epoch in range(num_epochs):
    # Set the model to training mode.
    # This enables layers like dropout and batch normalization to behave appropriately during training.
    roberta_model.train()

    # Initialize a variable to track the total loss for this epoch.
    total_loss = 0

    # Iterate over batches in the training DataLoader.
    # `tqdm` wraps the DataLoader to display a progress bar with the epoch number.
    for batch in tqdm(roberta_train_dataloader, desc=f"RoBERTa Epoch {epoch + 1}"):
        # Prepare inputs by moving relevant tensors to the specified device.
        # Only include 'input_ids' and 'attention_mask' as model inputs, excluding 'label'.
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}

        # Move the labels to the specified device.
        labels = batch["label"].to(device)

        # Forward pass: Feed inputs and labels to the model.
        # The model computes predictions and the loss (since labels are provided).
        outputs = roberta_model(**inputs, labels=labels)

        # Extract the loss from the model outputs.
        loss = outputs.loss

        # Backward pass: Compute gradients of the loss with respect to model parameters.
        loss.backward()

        # Update model parameters using the optimizer based on computed gradients.
        roberta_optimizer.step()

        # Clear accumulated gradients to prevent them from affecting the next batch.
        roberta_optimizer.zero_grad()

        # Add the batch loss (converted to a Python float) to the total loss.
        total_loss += loss.item()

    # Calculate and print the average loss for the epoch.
    # Divide total loss by the number of batches to get the average.
    print(f"RoBERTa Epoch {epoch + 1}, Average Loss: {total_loss / len(roberta_train_dataloader):.4f}")



Processing RoBERTa model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

RoBERTa Epoch 1, Average Loss: 0.3092


RoBERTa Epoch 2: 100%|██████████| 1250/1250 [30:01<00:00,  1.44s/it]


RoBERTa Epoch 2, Average Loss: 0.1813


RoBERTa Epoch 3: 100%|██████████| 1250/1250 [30:02<00:00,  1.44s/it]

RoBERTa Epoch 3, Average Loss: 0.1390


#### Evaluate RoBERTa Model on Test Set

In [ ]:
# --- Evaluate RoBERTa Model on Test Set ---
# Evaluate the RoBERTa model on the test dataset using a custom evaluation function.
# - `roberta_model`: The trained RoBERTa model (RobertaForSequenceClassification).
# - `roberta_test_dataloader`: PyTorch DataLoader containing the tokenized test dataset.
# - `device`: The hardware device (e.g., 'cuda' or 'cpu') where computations are performed.
# - `"RoBERTa"`: A string identifier for the model, used in the evaluation function for logging or reporting.
# The `evaluate_model` function (assumed to be defined elsewhere) likely computes metrics like accuracy, F1 score, etc.
roberta_results = evaluate_model(roberta_model, roberta_test_dataloader, device, "RoBERTa")

# Store the evaluation results in a dictionary called `results`.
# - `results` is assumed to be a dictionary defined elsewhere to store evaluation metrics for multiple models.
# - The key `"RoBERTa"` maps to the evaluation metrics returned by `evaluate_model`.
results["RoBERTa"] = roberta_results

# --- Save RoBERTa Model and Tokenizer ---
# Save the trained RoBERTa model to a local directory.
# - `save_pretrained("./roberta_ag_news")`: Saves the model's weights, configuration, and metadata to the specified directory.
# - The directory `./roberta_ag_news` will contain files like `pytorch_model.bin` (weights), `config.json` (model config), etc.
roberta_model.save_pretrained("./roberta_ag_news")

# Save the RoBERTa tokenizer to the same directory.
# - Saves the tokenizer's vocabulary, special tokens, and configuration.
# - The directory will include files like `vocab.json`, `merges.txt`, and `tokenizer_config.json`.
roberta_tokenizer.save_pretrained("./roberta_ag_news")

# Print a confirmation message to indicate that the model and tokenizer have been successfully saved.
print("RoBERTa model and tokenizer saved to ./roberta_ag_news")

 purest Evaluating RoBERTa: 100%|██████████| 82/82 [00:36<00:00,  2.27it/s]


RoBERTa model and tokenizer saved to ./roberta_ag_news


#### DistilBERT model

In [ ]:
# --- DistilBERT Model ---
print("Processing DistilBERT model...")

# Initialize the DistilBERT tokenizer
# Loads the pre-trained DistilBERT tokenizer from Hugging Face's transformers library
# 'distilbert-base-uncased' refers to the uncased version of DistilBERT, which is a smaller, faster, and lighter version of BERT
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Initialize the DistilBERT model for sequence classification
# Loads the pre-trained DistilBERT model configured for sequence classification
# 'num_labels=4' specifies that the model will classify inputs into 4 distinct categories
distilbert_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

# Move the model to the specified device (e.g., GPU or CPU)
# Ensures that model computations are performed on the appropriate hardware
distilbert_model.to(device)

# Tokenize the datasets for DistilBERT
# Applies the tokenize_function to the datasets using the DistilBERT tokenizer
# 'map' processes the dataset in batches for efficiency
# 'max_length=256' limits the input sequence length to 256 tokens to manage memory and computational load
distilbert_tokenized_datasets = datasets.map(lambda x: tokenize_function(x, distilbert_tokenizer, max_length=256), batched=True)

# Set the dataset format to PyTorch tensors
# Specifies that the dataset should output 'input_ids', 'attention_mask', and 'label' as PyTorch tensors
# This ensures compatibility with the PyTorch DataLoader and model inputs
distilbert_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Create a data loader for the training dataset
# DataLoader handles batching, shuffling, and loading of the training data
# 'batch_size=16' processes 16 samples per batch
# 'shuffle=True' randomizes the order of samples in each epoch to improve training
distilbert_train_dataloader = DataLoader(distilbert_tokenized_datasets["train"], batch_size=16, shuffle=True)

# Create a data loader for the test dataset
# Similar to the training DataLoader but without shuffling, as order doesn't matter for evaluation
distilbert_test_dataloader = DataLoader(distilbert_tokenized_datasets["test"], batch_size=16)

# Set up the optimizer for DistilBERT
# Uses AdamW (Adam with Weight Decay), a popular optimizer for transformer models
# 'lr=2e-5' sets the learning rate to 2e-5, a common choice for fine-tuning transformers
distilbert_optimizer = AdamW(distilbert_model.parameters(), lr=2e-5)

# Training loop for DistilBERT
# Iterates over the specified number of epochs (num_epochs)
for epoch in range(num_epochs):
    # Set the model to training mode
    # Enables layers like dropout and batch normalization to behave appropriately during training
    distilbert_model.train()

    # Initialize a variable to track the total loss for the epoch
    total_loss = 0

    # Iterate over batches in the training DataLoader
    # 'tqdm' adds a progress bar to visualize training progress
    for batch in tqdm(distilbert_train_dataloader, desc=f"DistilBERT Epoch {epoch + 1}"):
        # Move input tensors to the specified device
        # Filters batch to include only 'input_ids' and 'attention_mask' for model inputs
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}

        # Move labels to the specified device
        labels = batch["label"].to(device)

        # Forward pass: compute model outputs and loss
        # Passes inputs and labels to the model to get predictions and calculate the loss
        outputs = distilbert_model(**inputs, labels=labels)

        # Extract the loss from the model outputs
        loss = outputs.loss

        # Backward pass: compute gradients
        # Propagates the loss backward to compute gradients for model parameters
        loss.backward()

        # Update model parameters
        # Applies the gradients to update the model weights using the optimizer
        distilbert_optimizer.step()

        # Clear gradients
        # Resets the gradients to zero for the next batch to prevent accumulation
        distilbert_optimizer.zero_grad()

        # Accumulate the batch loss
        # Adds the batch loss to the total loss for the epoch
        total_loss += loss.item()
    print(f"DistilBERT Epoch {epoch + 1}, Average Loss: {total_loss / len(distilbert_train_dataloader):.4f}")



Processing DistilBERT model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

DistilBERT Epoch 1, Average Loss: 0.3145


DistilBERT Epoch 2: 100%|██████████| 1250/1250 [06:58<00:00,  2.98it/s]


DistilBERT Epoch 2, Average Loss: 0.1714


DistilBERT Epoch 3: 100%|██████████| 1250/1250 [06:58<00:00,  2.99it/s]

DistilBERT Epoch 3, Average Loss: 0.1031


#### Evaluate DistilBERT on the test set

In [ ]:
# Evaluate DistilBERT on the test set
# Calls the evaluate_model function to assess the model's performance on the test dataset
# Passes the model, test DataLoader, device, and model name for evaluation
# Stores the evaluation results in distilbert_results
distilbert_results = evaluate_model(distilbert_model, distilbert_test_dataloader, device, "DistilBERT")

# Store evaluation results
# Adds the DistilBERT evaluation results to a dictionary called 'results' under the key "DistilBERT"
results["DistilBERT"] = distilbert_results

# Save the DistilBERT model
# Saves the trained model's weights and configuration to the specified directory
# The model can be reloaded later for inference or further training
distilbert_model.save_pretrained("./distilbert_ag_news")

# Save the DistilBERT tokenizer
# Saves the tokenizer's configuration and vocabulary to the specified directory
# Ensures the tokenizer can be reloaded to preprocess text consistently
distilbert_tokenizer.save_pretrained("./distilbert_ag_news")

# Notify the user that the model and tokenizer have been saved
# Prints a confirmation message with the directory path where the model and tokenizer are saved
print("DistilBERT model and tokenizer saved to ./distilbert_ag_news")

 purest Evaluating DistilBERT: 100%|██████████| 82/82 [00:08<00:00,  9.27it/s]


DistilBERT model and tokenizer saved to ./distilbert_ag_news


#### GPT2 Model

In [ ]:
# --- GPT-2 Model ---
print("Processing GPT-2 model...")

# Initialize the GPT-2 tokenizer
# Loads the pre-trained GPT-2 tokenizer from Hugging Face's transformers library
# 'gpt2' refers to the base GPT-2 model, a transformer-based model designed for text generation
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Configure the pad token for GPT-2 tokenizer
# GPT-2 does not have a dedicated padding token; the end-of-sequence (EOS) token is used instead
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # GPT-2 doesn't have a pad token; use EOS token

# Initialize the GPT-2 model for sequence classification
# Loads the pre-trained GPT-2 model configured for sequence classification
# 'num_labels=4' specifies that the model will classify inputs into 4 distinct categories
gpt2_model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=4)

# Set the pad token ID in the model configuration
# Ensures the model recognizes the EOS token as the padding token during processing
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id  # Set pad token ID in model config

# Move the model to the specified device (e.g., GPU or CPU)
# Ensures that model computations are performed on the appropriate hardware
gpt2_model.to(device)

# Tokenize the datasets for GPT-2
# Applies the tokenize_function to the datasets using the GPT-2 tokenizer
# 'map' processes the dataset in batches for efficiency
# 'max_length=512' limits the input sequence length to 512 tokens, suitable for GPT-2's architecture
gpt2_tokenized_datasets = datasets.map(lambda x: tokenize_function(x, gpt2_tokenizer, max_length=512), batched=True)

# Set the dataset format to PyTorch tensors
# Specifies that the dataset should output 'input_ids', 'attention_mask', and 'label' as PyTorch tensors
# Ensures compatibility with the PyTorch DataLoader and model inputs
gpt2_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Create a data loader for the training dataset
# DataLoader handles batching, shuffling, and loading of the training data
# 'batch_size=16' processes 16 samples per batch
# 'shuffle=True' randomizes the order of samples in each epoch to improve training
gpt2_train_dataloader = DataLoader(gpt2_tokenized_datasets["train"], batch_size=16, shuffle=True)

# Create a data loader for the test dataset
# Similar to the training DataLoader but without shuffling, as order doesn't matter for evaluation
gpt2_test_dataloader = DataLoader(gpt2_tokenized_datasets["test"], batch_size=16)

# Set up the optimizer for GPT-2
# Uses AdamW (Adam with Weight Decay), a popular optimizer for transformer models
# 'lr=2e-5' sets the learning rate to 2e-5, a common choice for fine-tuning transformers
gpt2_optimizer = AdamW(gpt2_model.parameters(), lr=2e-5)

# Define the number of training epochs
# Specifies that the model will be trained for 3 epochs
num_epochs = 3

# Training loop for GPT-2
# Iterates over the specified number of epochs (3)
for epoch in range(num_epochs):

    # Set the model to training mode
    # Enables layers like dropout to behave appropriately during training
    gpt2_model.train()

    # Initialize a variable to track the total loss for the epoch
    total_loss = 0

    # Iterate over batches in the training DataLoader
    # 'tqdm' adds a progress bar to visualize training progress
    for batch in tqdm(gpt2_train_dataloader, desc=f"GPT-2 Epoch {epoch + 1}"):

        # Move input tensors to the specified device
        # Filters batch to include only 'input_ids' and 'attention_mask' for model inputs
        inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}

        # Move labels to the specified device
        labels = batch["label"].to(device

        # Forward pass: compute model outputs and loss
        # Passes inputs and labels to the model to get predictions and calculate the loss
        outputs = gpt2_model(**inputs, labels=labels)

        # Extract the loss from the model outputs
        loss = outputs.loss

        # Backward pass: compute gradients
        # Propagates the loss backward to compute gradients for model parameters
        loss.backward()

        # Update model parameters
        # Applies the gradients to update the model weights using the optimizer
        gpt2_optimizer.step()

        # Clear gradients
        # Resets the gradients to zero for the next batch to prevent accumulation
        gpt2_optimizer.zero_grad()

        # Accumulate the batch loss
        # Adds the batch loss to the total loss for the epoch
        total_loss += loss.item()
    print(f"GPT-2 Epoch {epoch + 1}, Average Loss: {total_loss / len(gpt2_train_dataloader):.4f}")



Processing GPT-2 model...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

GPT-2 Epoch 1: 100%|██████████| 1250/1250 [32:23<00:00,  1.56s/it]


GPT-2 Epoch 1, Average Loss: 0.3794


GPT-2 Epoch 2: 100%|██████████| 1250/1250 [32:28<00:00,  1.56s/it]


GPT-2 Epoch 2, Average Loss: 0.2025


GPT-2 Epoch 3: 100%|██████████| 1250/1250 [32:28<00:00,  1.56s/it]

GPT-2 Epoch 3, Average Loss: 0.1567


#### Evaluate GPT-2 on the test set

In [ ]:
# Evaluate GPT-2 on the test set
# Calls the evaluate_model function to assess the model's performance on the test dataset
# Passes the model, test DataLoader, device, and model name for evaluation
# Stores the evaluation results in gpt2_results
gpt2_results = evaluate_model(gpt2_model, gpt2_test_dataloader, device, "GPT-2")

# Store evaluation results
# Adds the GPT-2 evaluation results to a dictionary called 'results' under the key "GPT-2"
results["GPT-2"] = gpt2_results

# Save the GPT-2 model
# Saves the trained model's weights and configuration to the specified directory
# The model can be reloaded later for inference or further training
gpt2_model.save_pretrained("./gpt2_ag_news")

# Save the GPT-2 tokenizer
# Saves the tokenizer's configuration and vocabulary to the specified directory
# Ensures the tokenizer can be reloaded to preprocess text consistently
gpt2_tokenizer.save_pretrained("./gpt2_ag_news")

# Notify the user that the model and tokenizer have been saved
# Prints a confirmation message with the directory path where the model and tokenizer are saved
print("GPT-2 model and tokenizer saved to ./gpt2_ag_news")

 purest Evaluating GPT-2: 100%|██████████| 82/82 [00:36<00:00,  2.23it/s]


GPT-2 model and tokenizer saved to ./gpt2_ag_news


### Model Comparative Analysis

In [ ]:
# --- RoBERTa ---
print("Loading and evaluating RoBERTa...")
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta_ag_news")
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta_ag_news", num_labels=4)
roberta_model.to(device)
roberta_tokenized_datasets = datasets.map(
    lambda x: tokenize_function(x, roberta_tokenizer, max_length=512), batched=True
)
roberta_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
roberta_test_dataloader = DataLoader(roberta_tokenized_datasets["test"], batch_size=16)
results["RoBERTa"] = evaluate_model(roberta_model, roberta_test_dataloader, device, "RoBERTa")

# --- DistilBERT ---
print("Loading and evaluating DistilBERT...")
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert_ag_news")
distilbert_model = DistilBertForSequenceClassification.from_pretrained("distilbert_ag_news", num_labels=4)
distilbert_model.to(device)
distilbert_tokenized_datasets = datasets.map(
    lambda x: tokenize_function(x, distilbert_tokenizer, max_length=256), batched=True
)
distilbert_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
distilbert_test_dataloader = DataLoader(distilbert_tokenized_datasets["test"], batch_size=16)
results["DistilBERT"] = evaluate_model(distilbert_model, distilbert_test_dataloader, device, "DistilBERT")

# --- GPT-2 ---
print("Loading and evaluating GPT-2...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2_ag_news")
gpt2_model = GPT2ForSequenceClassification.from_pretrained("gpt2_ag_news", num_labels=4)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
gpt2_model.to(device)
gpt2_tokenized_datasets = datasets.map(
    lambda x: tokenize_function(x, gpt2_tokenizer, max_length=512), batched=True
)
gpt2_tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])
gpt2_test_dataloader = DataLoader(gpt2_tokenized_datasets["test"], batch_size=16)
results["GPT-2"] = evaluate_model(gpt2_model, gpt2_test_dataloader, device, "GPT-2")

# Convert results to DataFrame
results_df = pd.DataFrame.from_dict(results, orient="index").round(4)

# Display comparison table
print("\nModel Performance and Efficiency Comparison:")
print(results_df)

Loading and evaluating RoBERTa...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

 purest Evaluating RoBERTa: 100%|██████████████████████████████████████████████████████| 82/82 [18:16<00:00, 13.37s/it]


Loading and evaluating DistilBERT...


...

Loading and evaluating GPT-2...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

 purest Evaluating GPT-2: 100%|████████████████████████████████████████████████████████| 82/82 [22:03<00:00, 16.14s/it]


Model Performance and Efficiency Comparison:
            accuracy  precision  recall      f1  avg_inference_time  \
RoBERTa       0.9346     0.9351  0.9346  0.9348             13.3614   
DistilBERT    0.9162     0.9198  0.9162  0.9166              3.3319   
GPT-2         0.9338     0.9338  0.9338  0.9338             16.1359   

            total_inference_time avg_memory_usage  
RoBERTa                1095.6315             None  
DistilBERT              273.2177             None  
GPT-2                  1323.1474             None  


### RoBERTa is the most accurate at understanding text but takes longer, DistilBERT is the fastest but a bit less accurate, and GPT-2 is almost as accurate as RoBERTa but the slowest.

Note: Precision is how often the model is correct when it makes a guess, recall is how many of the right things it finds, and F1 is a score that shows how well it balances both.

## Appendix

#### Zip the saved files in Colab Cloud for download to local

In [ ]:
import zipfile
import os
from google.colab import files

# Function to zip a folder
def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(folder_path):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

# Paths
folder_to_zip = "/content/gpt2_ag_news"  # adjust this if your folder path is different
zip_output_path = "/content/gpt2_ag_news.zip"

#
# "/content/roberta_ag_news"
# "/content/roberta_ag_news.zip"

#
# "/content/distilbert_ag_news"
# "/content/distilbert_ag_news.zip"


#
# "/content/gpt2_ag_news"
# "/content/gpt2_ag_news.zip"


# Zip it
zip_folder(folder_to_zip, zip_output_path)

# Download
files.download(zip_output_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>